## 安装环境

In [18]:
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"

In [7]:
!pip install -q unsloth
!pip install -q datasets
!pip install -q trl
!pip install -q accelerate
!pip install --upgrade pip
!pip install -q bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ms-swift 4.5.2 requires peft<0.20,>=0.11, but you have peft 0.20.0 which is incompatible.
ms-funcodec 0.2.0 requires typeguard==2.13.3, but you have typeguard 4.6.0 which is incompatible.
ms-opencompass 0.1.6 requires numpy<2.0.0,>=1.23.4, but you have numpy 2.5.2 which is incompatible.
vllm 0.27.1 requires pydantic>=2.12.0, but you have pydantic 2.11.10 which is incompatible.
vllm 0.27.1 requires setuptools<81.0.0,>=77.0.3; python_version > "3.11", but you have setuptools 81.0.0 which is incompatible.
vllm 0.27.1 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
vllm 0.27.1 requires torch==2.13.0, but you have torch 2.12.1 which is incompatible.
vllm 0.27.1 requires torchvision==0.28.0, but you have torchvision 0.27.1 which is incompatible.
vllm 0.27.1 requires transformers>=5.5.3, b

## 引入所需要的包

In [1]:
import torch

from unsloth import FastLanguageModel

from trl import SFTTrainer

from transformers import TrainingArguments

from modelscope import snapshot_download

from datasets import load_from_disk

print(torch.cuda.get_device_name())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0914 09:45:36.442000 14900 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0914 09:45:36.472000 14900 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[xformers|WARNING]WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.12.1+cu130)
    Python  3.10.19 (you have 3.12.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for mo

Unsloth: causal_conv1d CUDA kernels not compatible with this GPU. Using PyTorch slow path for Mamba models.


Accessing `_lazy_load_causal_conv1d` from `.models.flava.image_processing_flava`. Returning `_lazy_load_causal_conv1d` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `_causal_conv1d_cache` from `.models.flava.image_processing_flava`. Returning `_causal_conv1d_cache` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `causal_conv1d_fn` from `.models.flava.image_processing_pil_flava`. Returning `causal_conv1d_fn` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `causal_conv1d_update` from `.models.flava.image_processing_pil_flava`. Returning `causal_conv1d_update` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `_lazy_load_causal_conv1d` from `.models.flava.image_processing_pil_flava`. Returning `_lazy_load_causal_conv1d` instead. Behavior may be different and this alias will be removed in future versi

Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/site-packages/unsloth/import_fixes.py:2124: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


NVIDIA A10


## 下载和加载模型

In [5]:
# model_dir = snapshot_download(
#     "Qwen/Qwen3.5-2B"
# )
model_dir = ".cache/modelscope/models/Qwen--Qwen3.5-2B/snapshots/master"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_dir,
    max_seq_length=2048,
    load_in_4bit=True,
)

2026-09-13 19:09:03,133 - modelscope - WARNING - The passed in library_name,library_version,user_agent,force_download,proxiesetag_timeout,headers,endpoint will not be used in modelscope.
/usr/local/lib/python3.12/site-packages/modelscope_hub/_download.py:596: FutureWarning: Environment variable 'INTRA_CLOUD_ACCELERATION_REGION' is deprecated, use 'MODELSCOPE_DOWNLOAD_INTRA_CLOUD_REGION' instead.
  self._download_with_resume(
2026-09-13 19:09:04,462 - modelscope - WARNING - The passed in library_name,library_version,user_agent,force_download,proxiesetag_timeout,headers,endpoint will not be used in modelscope.


==((====))==  Unsloth 2026.9.4: Fast Qwen3_5 patching. Transformers: 5.5.0. vLLM: 0.27.1.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


==((====))==  Unsloth 2026.9.4: Fast Qwen3_5 patching. Transformers: 5.5.0. vLLM: 0.27.1.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

## 加载 empathetic 数据集清洗、格式化

In [5]:
import re
from collections import defaultdict
from datasets import load_dataset, Dataset, DatasetDict

CACHE_PATH = "./facebook___empathetic_dialogues/default/0.0.0/d5b57ae707b0b9a384af8ed50c043c608d597ca7"
MIN_WORDS, MAX_WORDS = 2, 100

SYSTEM_PROMPT = "你是一个善于共情的对话助手。请根据用户说的话，生成一句温暖、贴切、有共情力的回复。"


def clean_text(text):
    if not isinstance(text, str):
        return ""
    for k, v in {"_comma_": ",", "_period_": ".", "_question_": "?",
                 "_exclamation_": "!", "_apostrophe_": "'", "_ellipsis_": "..."}.items():
        text = text.replace(k, v)
    return re.sub(r"\s+", " ", text).strip()


def is_valid(text):
    if not text or not isinstance(text, str):
        return False
    n = len(text.split())
    if n < MIN_WORDS or n > MAX_WORDS:
        return False
    alpha = sum(c.isalpha() or c.isspace() for c in text)
    return alpha / max(len(text), 1) >= 0.5


def build_pairs(split_data):
    convs = defaultdict(list)
    for s in split_data:
        convs[s["conv_id"]].append(s)
    pairs = []
    for turns in convs.values():
        turns = sorted(turns, key=lambda x: x["utterance_idx"])
        for a, b in zip(turns, turns[1:]):
            if b["utterance_idx"] != a["utterance_idx"] + 1:
                continue
            u, r = clean_text(a["utterance"]), clean_text(b["utterance"])
            if not is_valid(u) or not is_valid(r) or u.lower() == r.lower():
                continue
            pairs.append({"user_text": u, "reply": r})
    return pairs


def pairs_to_dataset(pairs, tokenizer):
    data = []
    for p in pairs:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": p["user_text"]},
            {"role": "assistant", "content": p["reply"]},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        data.append({"text": text})
    return Dataset.from_list(data)


def get_dataset(tokenizer):
    ds = load_dataset(CACHE_PATH)
    return DatasetDict({
        "train": pairs_to_dataset(build_pairs(ds["train"]), tokenizer),
        "validation": pairs_to_dataset(build_pairs(ds["validation"]), tokenizer),
    })

In [7]:
dataset = get_dataset(tokenizer)
dataset['train'][0]

{'text': '<|im_start|>system\n你是一个善于共情的对话助手。请根据用户说的话，生成一句温暖、贴切、有共情力的回复。<|im_end|>\n<|im_start|>user\nI remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people, we felt like the only people in the world.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nWas this a friend you were in love with, or just a best friend?<|im_end|>\n'}

# 开始微调 QLoRa

In [ ]:
# 挂上 QLoRa
qlora_model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


/usr/local/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [36]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

dataset = get_dataset(tokenizer)
qlora_model.config.use_cache = False

class LossLogger(TrainerCallback):
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.steps = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.train_losses.append(logs["loss"])
            self.steps.append(state.global_step)
        if logs and "eval_loss" in logs:
            self.eval_losses.append(logs["eval_loss"])

trainer = SFTTrainer(
    model=qlora_model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    args=SFTConfig(
        output_dir="./qwen-empathetic-qlora-checkpoints",
        dataset_text_field="text",

        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=4,

        max_length=512,
        packing=False,
        gradient_checkpointing=False,

        learning_rate=2e-4,
        num_train_epochs=2,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",

        weight_decay=0.05,
        bf16=True,
        optim="adamw_8bit",

        dataloader_num_workers=4,
        dataloader_pin_memory=True,

        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        save_total_limit=6,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )
)

trainer.add_callback(LossLogger())
trainer.train()

# 保存训练后的模型
trainer.save_model("./qwen35-empathetic-qlora")
tokenizer.save_pretrained("./qwen35-empathetic-qlora")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/58212 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/9186 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 58,212 | Num Epochs = 2 | Total steps = 7,278
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Step,Training Loss,Validation Loss
100,1.127919,1.390398
200,1.304506,1.337701
300,1.311624,1.327869
400,1.244355,1.324251
500,1.287702,1.318351
600,1.310341,1.317703
700,1.263613,1.317868
800,1.237247,1.315596
900,1.264459,1.317221
1000,1.268384,1.316276


Unsloth: Restored added_tokens_decoder metadata in ./qwen-empathetic-qlora-checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-empathetic-qlora-checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-empathetic-qlora-checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-empathetic-qlora-checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-empathetic-qlora-checkpoints/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-empathetic-qlora-checkpoints/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-empathetic-qlora-checkpoints/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./qwen-empathetic-qlora-checkpoints/checkpoint-900/tokenizer_config.json.


: 

# 推理

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
model_dir = ".cache/modelscope/models/Qwen--Qwen3.5-2B/snapshots/master"
# 加载基座模型
tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
test_model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [3]:
# 加载 LoRA 适配器
test_model = PeftModel.from_pretrained(
    test_model,
    "qwen-empathetic-qlora-checkpoints/checkpoint-3600",
    adapter_name="first" 
)

# 加载第二个 Lora
test_model.load_adapter(
    "qwen-esconv-qlora-checkpoints/checkpoint-171",
    adapter_name="second"
)
# 激活多个适配器
test_model.base_model.set_adapter(["first", "second"])

/usr/local/lib/python3.12/site-packages/peft/peft_model.py:665: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.mlp.gate_proj.lora_A.first.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.first.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.first.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.first.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_A.first.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_B.first.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_A.first.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_B.first.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_A.first.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_B.first.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_A.first.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_B.first.weight', 'base_model.model.model.layers.2.mlp.gate_pro

In [4]:
# 推理模式
test_model.eval()
SYSTEM_PROMPT = "你是一个善于共情的对话助手。请根据用户说的话，生成一句温暖、贴切、有共情力的回复。"
# 构造输入
prompt = "我有点难过"

messages = [
    {"role":"system","content":SYSTEM_PROMPT},
    {"role": "user", "content": prompt}
]

# 用 tokenizer 自带的 chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(test_model.device)

with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(response)

抱抱你，难过的时候不用急着好起来，先允许自己此刻的脆弱，好吗？🫂


## 基础模型测试对比

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# 推理模式
base_model.eval()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 2048)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
          (norm): FusedRMSNormGated(128, eps=1e-06, activation=silu)
          (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (in_proj_qkv): Linear(in_features=2048, out_features=6144, bias=False)
          (in_proj_z): Linear(in_features=2048, out_features=2048, bias=False)
          (in_proj_b): Linear(in_features=2048, out_features=16, bias=False)
          (in_proj_a): Linear(in_features=2048, out_features=16, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
 

In [10]:
# 构造输入
prompt = "我有点难过"

messages = [
    {"role":"system","content":SYSTEM_PROMPT},
    {"role": "user", "content": prompt}
]

# 用 tokenizer 自带的 chat template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to(base_model.device)

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(response)

抱抱你，摸摸头。难过的时候真的很难受，但你的感受是完全被接纳的，不用急着去“好起来”。先深呼吸一下，允许自己此刻就这样待一会儿，好吗？🫂
